# M6-T6: NF4-base extraction probe for Llama-3.1-8B (D-004, D-098)

**Notebook settings:** Accelerator **GPU T4 x2** · Internet **on** · Secrets: **HF_TOKEN**. Run as **Save Version → Save & Run All**. Expected ≈ 1–1.5 h.

Re-extracts `ref`, `a100` and `a000` of Llama-3.1-8B on an **NF4** base (the QLoRA training base) instead of fp16, with `configs/m6_nf4_llama3.1-8b.yaml` (same train keys: the trained adapters are reused, never retrained), then computes ΔB (`deltab`) and syncs `results/m6_nf4_llama3.1-8b/`. Run `m6_analysis.ipynb` afterwards: its `nf4.csv` compares ΔB_nf4 with the fp16 ΔB of M4.

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "claude/kaggle-bias-detection-plan-h9es1w"  # a branch, tag or FULL 40-char SHA
WORK = "/kaggle/working/rbbd"
NAME = "m6_nf4_llama3.1-8b"
import os, time
SESSION_T0 = time.time()

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
# B-032 / D-087: the Kaggle image runs Python 3.13 and vllm 0.10.1.1 has no 3.13 wheels.
# The pinned stack goes into a Python 3.12 venv on ephemeral disk. Every uv call names its
# target interpreter explicitly, and the image's own UV_* settings are cleared first: they
# made uv install into the system Python (B-032, second run). Putting the venv's bin/ first
# on PATH makes every `!python ...` line and the stage subprocesses use it. This kernel
# only imports rbbd's light modules (config, runner, manifests) from src/.
import os, shutil, subprocess

def sh(*cmd, tail=25):
    # Run a setup command; show the end of its output; stop the notebook if it fails.
    r = subprocess.run(list(cmd), capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    print(f"$ {' '.join(cmd[:4])} ... -> exit {r.returncode}")
    if tail:
        print("\n".join(out[-tail:]))
    if r.returncode:
        raise RuntimeError(f"setup step failed: {' '.join(cmd[:4])}")
    return r.stdout

uv_vars = sorted(k for k in os.environ if k.startswith("UV_"))
print("image UV_* settings cleared:", uv_vars)  # names only
for k in uv_vars:
    del os.environ[k]
EPH = next((p for p in ("/kaggle/tmp", "/kaggle/temp") if os.path.isdir(p)), "/tmp")
VENV = f"{EPH}/rbbd_py312"
PY = f"{VENV}/bin/python"
os.environ.update(UV_CACHE_DIR=f"{EPH}/uv_cache", UV_LINK_MODE="copy",
                  UV_PYTHON_INSTALL_DIR=f"{EPH}/uv_python")
sh("pip", "install", "-q", "uv==0.8.17")
UV = shutil.which("uv")
sh(UV, "python", "install", "3.12")
sh(UV, "venv", "--python-preference", "only-managed", "--python", "3.12", VENV)
sh(PY, "--version")
sh(UV, "pip", "install", "--python", PY, "-e", ".[train,bench,dev]")
os.environ["VIRTUAL_ENV"] = VENV
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
assert shutil.which("python") == PY, shutil.which("python")
with open("/kaggle/working/pip_freeze_m6.txt", "w") as fh:
    fh.write(sh(UV, "pip", "freeze", "--python", PY, tail=0))
# Preflight (B-012, D-049): stop here, not inside a stage, if the pinned stack does not import.
print(sh(PY, "-c", "import sys, torch, torchvision, transformers, vllm; print('preflight ok', "
         "sys.version.split()[0], torch.__version__, torchvision.__version__, "
         "transformers.__version__, vllm.__version__)"))

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# Restore the trained adapters, check every endpoint exists (never train here), extract, ΔB, sync.
import json, shutil
from pathlib import Path
from rbbd import config as _config, runner
from rbbd.utils import manifest as mf
ROOT = Path("/kaggle/working/artifacts")
CFG = f"configs/{NAME}.yaml"
cfg = _config.load(CFG)
SLUG, RUN = cfg["models"][0]["slug"], cfg["run_name"]
!python -m rbbd.cli restore --path train/{SLUG}
!python -m rbbd.cli run --config {CFG} --stages sentences,ftdata
from rbbd.finetune import sft
keys = runner.stage_run_keys(cfg)
ft = mf.read(mf.manifest_path(ROOT, "ftdata", keys["ftdata"]))
jobs = sft.plan_jobs(cfg, ROOT, list(ft.output_hashes))
missing = [str(j.out_dir) for j in jobs if not (j.out_dir / "done.json").exists()]
if missing:
    raise RuntimeError(f"endpoints not found after restore (train keys changed?): {missing}")
t0 = time.time()
!python -m rbbd.cli run --config {CFG} --stages sentences,ftdata,train,embed,deltab
out = {"wall_s": round(time.time() - t0, 1)}
for stage in ("embed", "deltab"):
    m = mf.read(mf.manifest_path(ROOT, stage, keys[stage]))
    out[f"{stage}_complete"] = bool(m and m.complete)
tdir = ROOT / "embed" / keys["embed"] / "timing.json"
if tdir.exists():
    t = json.loads(tdir.read_text())["models"][SLUG]
    out["embed"] = {"load_seconds": t["load_seconds"],
                    "checkpoints": {k: {kk: v.get(kk) for kk in ("seconds", "cache_hit", "peak_mem_gib")}
                                    for k, v in t["checkpoints"].items()}}
s = ROOT / "deltab" / keys["deltab"] / "sanity.json"
out["sanity"] = json.loads(s.read_text()) if s.exists() else None
res = Path(WORK) / "results" / RUN
if res.is_dir():
    shutil.copytree(res, ROOT / "results" / RUN, dirs_exist_ok=True)
for path in ["manifests", "embed", "deltab", f"results/{RUN}"]:
    if (ROOT / path).exists():
        !python -m rbbd.cli sync --path {path}
print(json.dumps(out, indent=1, default=str))
!python -m rbbd.cli sync --path env/$RBBD_SESSION_ID
print("session hours:", round((time.time() - SESSION_T0) / 3600, 2))